# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Yingying LIU

**ID**: 47236769

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()
Pkg.add("CSV")

  Activating project at `c:\Users\Lyy\hw05`


In [12]:
using JuMP
using HiGHS
using CSV
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

    Updating registry at `C:\Users\Lyy\.julia\registries\General.toml`
   Resolving package versions...
    Updating `C:\Users\Lyy\hw05\Project.toml`
  [336ed68f] + CSV v0.10.15
    Updating `C:\Users\Lyy\hw05\Manifest.toml`
  [336ed68f] + CSV v0.10.15
  [48062228] + FilePathsBase v0.9.24
  [ea10d353] + WeakRefStrings v1.4.2
  [76eceee3] + WorkerUtilities v1.6.1


## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.



-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1
Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

#### Answer 1.1
1. Calculate overall recycling fraction R,<br>
   $R=\sum_{component}\frac{mass\%_{recycle}}{100}$<br>
   - Paper: 0.4*0.55=0.22
   - Yard: 0.18*0.4=0.072
   - Glass: 0.04*0.6=0.024
   - Aluminum: 0.02*0.8=0.016
   - Ferrous: 0.02*0.75=0.015
   - Wood: 0.05*0.3=0.015
   - Other Metal: 0.01*0.5=0.005
   - Plastics: 0.05*0.15=0.0075
   - Textiles: 0.03*0.1=0.003
   - R=0.22+0.072+0.024+0.016+0.015+0.015+0.005+0.0075+0.003=0.3775<br>
2. Calculate overall ash fraction A,<br>
   $A=\sum_{component}\frac{mass\%_{ash}}{100}$<br>
   - Food: 0.15*0.08=0.012
   - Paper: 0.4*0.07=0.028
   - Plastics: 0.05*0.05=0.0025
   - Textiles: 0.03*0.1=0.003
   - Rubber: 0.02*0.15=0.003
   - Wood: 0.05*0.02=0.001
   - Yard: 0.18*0.02=0.0036
   - Glass: 0.04*1=0.04
   - Ferrous: 0.02*1=0.02
   - Aluminum: 0.02*1=0.02
   - Other Metal: 0.01*1=0.01
   - Miscellaneous: 0.03*0.7=0.021
   - A=0.012+0.028+0.0025+0.003+0.003+0.001+0.0036+0.04+0.02+0.02+0.01+0.021=0.1641<br>

Therefore, the recycling fraction is **37.75%** and ash fraction is **16.41%**.

#### Problem 1.2
What are the decision variables for your optimization problem? Provide
notation and variable meaning.

#### Answer 1.2
Decision variables:
- $x_{c,f}$, where $c \in \{1,2,3\}$ and $f \in \{LF, MRF, MTE\}$. $x_{c,f}$ represents the amount of water (Mg/day) sent from city $c$ to facility $f$.
- $y_f$, $y_f \in \{0,1\}$. $y_f$ is a binary decision variable indicating whether the facility $f$ is opened.$y_f=1$ if $f$ is implemented, $y_f=0$ if $f$ is not implemented.

#### Problem 1.3
Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

#### Answer 1.3
The objective function is to **minimize the total cost**, which includes fixed cost, tipping cost, transport cost, MRF recycling processing cost, and transport & tipping cost for MRF residuals when they are sent to LF or WTE.

- Fixed cost: <br>
  $\sum_f F_f y_f$, $F_f$=[2000, 1500, 2500].<br>
- Tipping cost: <br>
  $\sum_c\sum_f p_f x_{c,f}$, $p_f$=[50, 7, 60].<br>
- Transport cost from city to facility:<br>
  $\sum_c \sum_f t d_{c,f} x_{c,f}$, <br>
  where t is transportation cost \$1.5/Mg-km, d is distance between city $c$ and facility $f$.<br>
- MRF recycling process cost: <br>
  $c_{recycle}R\sum_c x_{c,MRF}$,<br>
  where c is the recycling cost $40/Mg, R is the combined recyclable fraction of MSW. <br>
- Transport cost and tipping from MRF to LF or WTE: <br>
  $\sum_{dest \in \{LF,WTE\}}(p_{destination}+t d_{MRF,destination}r)$,<br>
  where r represents the amount of residual waste leaving the MRF and transported to the final disposal facility.

  Therefore, the objective function is, <br>
  min $\sum_f F_f y_f + \sum_c \sum_f (P_f+t d_{c,f})x_{c,f} +c_{recycle}R\sum_c x_{c,MRF} +   \sum_{dest \in \{LF,WTE\}}(p_{destination}+t d_{MRF,destination}r)$.

#### Problem 1.4
Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

#### Answer 1.4
- City mass balance constraint: all waste must be shipped somewhere.<br>
  $x_{c,LF}+x_{c,MRF}+x_{c,WTE}=W_c$<br>
- MRF residual mass balance constraint: the mass leaving the MRF as residuals equals the non-recycled fraction of the MRF inflow.
  $r_{MRF->LF}+r_{MRF->WTE}=(1-R)\sum_c x_{c,MRF}$.
- Facility capacities constraint:<br>
  - MRF throughput:<br>
    $\sum_c x_{c,MRF} \le Cap_{MRF} y_{MRF}$,<br>
  - LF capacity:<br>
    $\sum_c x_{c,LF}+r_{MRF->LF} \le Cap_{LF} y_{LF}$,<br>
  - WTE capacity:<br>
    $\sum_c x_{c,WTE}+r_{MRF->WTE} \le Cap_{WTE} y_{WTE}$.<br>
- Nonnecativity constraint:<br>
  $x_{c,f} \ge 0$, $r_{MRF->LF}\ge 0$, $r_{MRF->WTE} \ge 0$.   

#### Problem 1.5
Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

#### Answer 1.5
The optimal objective value is **$25,750 per day**. In optimal senario, 200 Md will be processed with LF, and 110 Md will be processed with WTE. MRF will not be used.

In [ ]:
# -------------------------
# Data
# -------------------------
cities = [1,2,3]
W = Dict(1=>100.0, 2=>90.0, 3=>120.0)  # Mg/day per city

facilities = [:LF, :MRF, :WTE]
capacity = Dict(:LF=>200.0, :MRF=>350.0, :WTE=>210.0)
fixed_cost = Dict(:LF=>2000.0, :MRF=>1500.0, :WTE=>2500.0)
tipping = Dict(:LF=>50.0, :MRF=>7.0, :WTE=>60.0)

c_recycle = 40.0           # $/Mg recycled
t_per_Mg_km = 1.5          # $ per Mg-km

# city -> facility distances (km)
d = Dict(
    1 => Dict(:LF=>5.0,  :MRF=>30.0, :WTE=>15.0),
    2 => Dict(:LF=>15.0, :MRF=>25.0, :WTE=>10.0),
    3 => Dict(:LF=>13.0, :MRF=>45.0, :WTE=>20.0)
)

# MRF -> destinations distances (km)
d_mrf = Dict(:LF=>32.0, :WTE=>15.0)

# Composition table (mass %, ash %, MRF recycling rate %)
# Format: (mass_pct, ash_pct, mrf_rate_pct)
components = Dict(
  "Food Wastes" => (15.0, 8.0, 0.0),
  "Paper & Cardboard" => (40.0, 7.0, 55.0),
  "Plastics" => (5.0, 5.0, 15.0),
  "Textiles" => (3.0, 10.0, 10.0),
  "Rubber, Leather" => (2.0, 15.0, 0.0),
  "Wood" => (5.0, 2.0, 30.0),
  "Yard Wastes" => (18.0, 2.0, 40.0),
  "Glass" => (4.0, 100.0, 60.0),
  "Ferrous" => (2.0, 100.0, 75.0),
  "Aluminum" => (2.0, 100.0, 80.0),
  "Other Metal" => (1.0, 100.0, 50.0),
  "Miscellaneous" => (3.0, 70.0, 0.0)
)

# -------------------------
# Compute R and A_orig
# -------------------------
R = 0.0
A_orig = 0.0
for (name, (mass_pct, ash_pct, mrf_rate_pct)) in components
    mf = mass_pct / 100.0
    R += mf * (mrf_rate_pct / 100.0)
    A_orig += mf * (ash_pct / 100.0)
end

println("Overall recycling fraction R = $(round(R, digits=5)) (", round(R*100, digits=2), "%)")
println("Overall original ash fraction A_orig = $(round(A_orig, digits=5)) (", round(A_orig*100, digits=2), "%)\n")

# -------------------------
# Build JuMP model
# -------------------------
model = Model(HiGHS.Optimizer)

# Decision variables
# x[c,f] >= 0 : flow from city c to facility f
@variable(model, x[cities, facilities] >= 0)

# residuals from MRF to LF and WTE
@variable(model, r_mrf_to_LF >= 0)
@variable(model, r_mrf_to_WTE >= 0)

# binary facility open variables
@variable(model, y[f in facilities], Bin)

# -------------------------
# Objective terms
# -------------------------
# fixed costs
fixed_term = sum(fixed_cost[f]*y[f] for f in facilities)

# city->facility variable costs: tipping + transport
var_term = 0.0
for c in cities, f in facilities
    var_term += (tipping[f] + t_per_Mg_km * d[c][f]) * x[c,f]
end

# recycling processing cost (applies to recycled portion R * inflow to MRF)
recycle_term = c_recycle * R * sum(x[c,:MRF] for c in cities)

# MRF residual transport + tipping costs
mrf_res_term = (tipping[:LF] + t_per_Mg_km * d_mrf[:LF]) * r_mrf_to_LF +
               (tipping[:WTE] + t_per_Mg_km * d_mrf[:WTE]) * r_mrf_to_WTE

@objective(model, Min, fixed_term + var_term + recycle_term + mrf_res_term)

# -------------------------
# Constraints
# -------------------------
# 1) City mass balance: all waste must be sent somewhere
for c in cities
    @constraint(model, sum(x[c,f] for f in facilities) == W[c])
end

# 2) MRF residual balance:
# r_mrf_to_LF + r_mrf_to_WTE == (1-R) * sum_c x[c,MRF]
@constraint(model, r_mrf_to_LF + r_mrf_to_WTE == (1.0 - R) * sum(x[c,:MRF] for c in cities))

# 3) Facility capacities (link to y using capacities)
# MRF capacity
@constraint(model, sum(x[c,:MRF] for c in cities) <= capacity[:MRF] * y[:MRF])

# LF capacity: direct city->LF flows plus MRF->LF residual cannot exceed LF capacity if open
@constraint(model, sum(x[c,:LF] for c in cities) + r_mrf_to_LF <= capacity[:LF] * y[:LF])

# WTE capacity: direct city->WTE flows plus MRF->WTE residual cannot exceed WTE capacity if open
@constraint(model, sum(x[c,:WTE] for c in cities) + r_mrf_to_WTE <= capacity[:WTE] * y[:WTE])

# 4) Prevent flows to closed facilities implicitly handled by capacity constraints:
# (if y[f]==0 then sum flows to f <= 0)
# (Additionally we might want at least one facility open; optional)
@constraint(model, y[:LF] + y[:MRF] + y[:WTE] >= 1)

# -------------------------
# Solve
# -------------------------
optimize!(model)

status = termination_status(model)
ps = primal_status(model)
println("Solve status: ", status, " / primal_status: ", ps)

if status == MOI.OPTIMAL || status == MOI.LOCALLY_SOLVED
    obj = objective_value(model)
    println("\nOptimal objective (total daily cost): \$", round(obj, digits=2))
    println("Facility open decisions:")
    for f in facilities
        println("  ", f, " open y = ", Int(round(value(y[f]))))
    end

    println("\nCity -> Facility flows (Mg/day):")
    for c in cities
        for f in facilities
            println("  City ", c, " -> ", f, " : ", round(value(x[c,f]), digits=2))
        end
    end

    println("\nMRF residual flows (Mg/day):")
    println("  MRF -> LF  : ", round(value(r_mrf_to_LF), digits=2))
    println("  MRF -> WTE : ", round(value(r_mrf_to_WTE), digits=2))

    # Summaries
    LF_recv = sum(value(x[c,:LF]) for c in cities) + value(r_mrf_to_LF)
    MRF_recv = sum(value(x[c,:MRF]) for c in cities)
    WTE_recv = sum(value(x[c,:WTE]) for c in cities) + value(r_mrf_to_WTE)
    println("\nTotal receipts (Mg/day): LF=", round(LF_recv,digits=2),
            ", MRF=", round(MRF_recv,digits=2), ", WTE=", round(WTE_recv,digits=2))
else
    println("Model not solved to optimality. Status: ", status)
end


Overall recycling fraction R = 0.3775 (37.75%)
Overall original ash fraction A_orig = 0.1641 (16.41%)

Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 8 rows; 14 cols; 31 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [6e-01, 4e+02]
  Cost    [6e+01, 2e+03]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+02]
Presolving model
8 rows, 14 cols, 31 nonzeros  0s
8 rows, 14 cols, 31 nonzeros  0s
Presolve reductions: rows 8(-0); columns 14(-0); nonzeros 31(-0) - Not reduced

Solving MIP model with:
   8 rows
   14 cols (3 binary, 0 integer, 0 implied int., 11 continuous, 0 domain fixed)
   31 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p =

#### Problem 1.6
Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

#### Answer 1.6
- **WRF** is not used. And this optimal result **makes sense**.<br>
- The reason why WRF is not used is that except tipping fee, **it still has high recycling fee ($40/Mg) and residual transport fee**. Only 37.75% of MRF is recycled, the 62.25% residual costs high transport and tipping fee from MRF site to other facilities.<br>
- Take city 2 as example (same for city 1 and city 3), <br>
  - In Case: City2 -> MRF<br>
    - MRF then residual -> WTE<br>
      City2 -> MRF transport = 25 km × 1.5 = $37.50<br>
    - MRF tipping fee on inflow = $7.00<br>
      Recycling processing (per inflow) = $15.10<br>
    - Residual fraction = 1 − R = 0.6225<br>
      MRF -> WTE transport = 15 km × 1.5 = 22.5; tipping at WTE = 60 -> dest cost = 22.5 + 60 = $82.50<br>
    - Residual cost per MRF inflow = 0.6225 × 82.5 = $51.37
      Total per Mg of city waste sent to MRF (with residual to WTE) ≈ 37.50 + 7.00 + 15.10 + 51.37 = $111.97 / Mg<br>
  - In Case: City2 -> direct to WTE<br>
    As earlier: 60 + 10×1.5 = 60 + 15 = $75 / Mg<br>
  - In Case: City2 -> direct to LF<br>
    As earlier: 50 + 15×1.5 = 50 + 22.5 = $72.50 / Mg<br>
- Therefore, sending to MRF costs much more money than direct to LF ($75 / Mg) or WTE ($72.50 / Mg).<br>

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data, including ramping constraints for each generator, is provided in \`data/generators.csv.’ 

In period 1, the demand is $d_1 = 1100 \text{MW}$.

In period 2, the demand is projected to be $d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind capacity factor is $0.45$. 

But in the second period, there is some uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are $0.75$ and $0.5$.

Your goal is to identify how to dispatch your generators to minimize the cost of meeting demand.

#### Problem 2.1
Draw a scenario tree for this problem.

#### Answer 2.1
So four senarios with probabilities:
$p=(0.175, 0.075, 0.225, 0.525)$.

#### Problem 2.2
Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

#### Answer 2.2
Expected cost = 17720.0<br>

--- Period 1 ---<br>
Biomass         : 0.0<br>
Hydroelectric   : 0.0<br>
Geothermal      : 0.0<br>
NG_CCGT         : 220.0<br>
NG_CT           : 100.0<br>
Wind            : 0.0<br>
Solar           : 195.0<br>
Solar1 = 450.0  (max =450.0)<br>
Wind1  = 135.0  (max =135.0)<br>

--- Period 2: Scenario s1  prob=0.525 ---<br>
Solar2 = 475.0  (max=475.0)<br>
Wind2  = 120.0  (max=120.0)<br>
Biomass         : 0.0<br>
Hydroelectric   : 0.0<br>
Geothermal      : 0.0<br>
NG_CCGT         : 220.0<br>
NG_CT           : 100.0<br>
Wind            : 0.0<br>
Solar           : 285.0<br>

--- Period 2: Scenario s2  prob=0.225 ---<br>
Solar2 = 375.0  (max=375.0)<br>
Wind2  = 150.0  (max=150.0)<br>
Biomass         : 0.0<br>
Hydroelectric   : 0.0<br>
Geothermal      : 0.0<br>
NG_CCGT         : 220.0<br>
NG_CT           : 100.0<br>
Wind            : 0.0<br>
Solar           : 355.0<br>

--- Period 2: Scenario s3  prob=0.175 ---<br>
Solar2 = 475.0  (max=475.0)<br>
Wind2  = 120.0  (max=120.0)<br>
Biomass         : 0.0<br>
Hydroelectric   : 0.0<br>
Geothermal      : 0.0<br>
NG_CCGT         : 220.0<br>
NG_CT           : 100.0<br>
Wind            : 85.0<br>
Solar           : 500.0<br>

--- Period 2: Scenario s4  prob=0.075 ---<br>
Solar2 = 375.0  (max=375.0)<br>
Wind2  = 150.0  (max=150.0)<br>
Biomass         : 0.0<br>
Hydroelectric   : 0.0<br>
Geothermal      : 0.0<br>
NG_CCGT         : 220.0<br>
NG_CT           : 100.0<br>
Wind            : 155.0<br>
Solar           : 500.0<br>

In [14]:
# ---------------------------------------------------------
# Load generator data from CSV
# ---------------------------------------------------------
genfile = "data/generators.csv"   # <- change path if needed
df = CSV.read(genfile, DataFrame)

# Trim spaces from column names and strings
for c in names(df)
    if eltype(df[!, c]) <: AbstractString
        df[!, c] = strip.(df[!, c])
    end
end

# Fix plant names (replace spaces for symbols)
df[!, :Plant] = replace.(df.Plant, " " => "_")

gens = df.Plant
Pmin = Dict(df.Plant .=> df.Pmin)
Pmax = Dict(df.Plant .=> df.Pmax)
VarCost = Dict(df.Plant .=> df.VarCost)
Ramp = Dict(df.Plant .=> df.Ramp)

# Identify renewable plants
solar_name = "Solar"
wind_name  = "Wind"

CapSolar = Pmax[solar_name]
CapWind  = Pmax[wind_name]

# ---------------------------------------------------------
# Scenario tree & demand/renewable uncertainty
# ---------------------------------------------------------
d1 = 1100.0
cf_solar_1 = 0.90
cf_wind_1  = 0.45

# 4-scenario tree
scenarios = [:s1, :s2, :s3, :s4]
p = Dict(:s1=>0.525, :s2=>0.225, :s3=>0.175, :s4=>0.075)

d2 = Dict(:s1=>1200.0, :s2=>1200.0, :s3=>1500.0, :s4=>1500.0)

cf_solar_2 = Dict(:s1=>0.95, :s2=>0.75, :s3=>0.95, :s4=>0.75)
cf_wind_2  = Dict(:s1=>0.40, :s2=>0.50, :s3=>0.40, :s4=>0.50)

# ---------------------------------------------------------
# JuMP Model
# ---------------------------------------------------------
model = Model(HiGHS.Optimizer)

# Period 1 generation
@variable(model, Pmin[g] <= g1[g in gens] <= Pmax[g])

# Period 1 renewable injections
@variable(model, 0 <= solar1 <= CapSolar * cf_solar_1)
@variable(model, 0 <= wind1  <= CapWind  * cf_wind_1)

# Period 2 generation
@variable(model, 0 <= g2[g in gens, s in scenarios] <= Pmax[g])

# Period 2 renewable injections
@variable(model, 0 <= solar2[s in scenarios] <= CapSolar * cf_solar_2[s])
@variable(model, 0 <= wind2[s in scenarios]  <= CapWind  * cf_wind_2[s])

# Objective
@objective(model, Min,
    sum(VarCost[g] * g1[g] for g in gens) +
    sum(p[s] * sum(VarCost[g] * g2[g,s] for g in gens) for s in scenarios)
)

# Period 1 demand balance
@constraint(model, sum(g1[g] for g in gens) + solar1 + wind1 == d1)

# Period 2 demand balance
for s in scenarios
    @constraint(model, sum(g2[g,s] for g in gens) + solar2[s] + wind2[s] == d2[s])
end

# Ramp constraints
for g in gens, s in scenarios
    @constraint(model, g2[g,s] - g1[g] <= Ramp[g])
    @constraint(model, g1[g] - g2[g,s] <= Ramp[g])
end

# Minimum output in period 2
for g in gens, s in scenarios
    @constraint(model, g2[g,s] >= Pmin[g])
end

# ---------------------------------------------------------
# Solve
# ---------------------------------------------------------
optimize!(model)

println("Status = ", termination_status(model))
println("\nExpected cost = ", objective_value(model))

println("\n--- Period 1 ---")
for g in gens
    println(rpad(g,15), " : ", round(value(g1[g]), digits=3))
end
println("Solar1 = ", value(solar1), "  (max =", CapSolar*cf_solar_1, ")")
println("Wind1  = ", value(wind1),  "  (max =", CapWind*cf_wind_1, ")")

for s in scenarios
    println("\n--- Scenario ", s, "  prob=", p[s], " ---")
    println("Solar2 = ", value(solar2[s]), "  (max=", CapSolar*cf_solar_2[s], ")")
    println("Wind2  = ", value(wind2[s]),  "  (max=", CapWind*cf_wind_2[s], ")")
    for g in gens
        println(rpad(g,15), " : ", round(value(g2[g,s]), digits=3))
    end
end


Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
LP has 89 rows; 45 cols; 185 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [4e-01, 4e+02]
  Bound   [1e+02, 5e+02]
  RHS     [1e+02, 2e+03]
Presolving model
13 rows, 40 cols, 56 nonzeros  0s
9 rows, 20 cols, 28 nonzeros  0s
4 rows, 5 cols, 8 nonzeros  0s
Presolve reductions: rows 4(-85); columns 5(-40); nonzeros 8(-177) 
Solving the presolved LP

Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     :  1.7720000000e+04
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.02
Status = 

┌ Warning: thread = 1 warning: parsed expected 6 columns, but didn't reach end of line around data row: 3. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\Lyy\.julia\packages\CSV\XLcqT\src\file.jl:593
┌ Warning: thread = 1 warning: only found 6 / 7 columns around data row: 4. Filling remaining columns with `missing`
└ @ CSV C:\Users\Lyy\.julia\packages\CSV\XLcqT\src\file.jl:592
┌ Warning: thread = 1 warning: only found 6 / 7 columns around data row: 5. Filling remaining columns with `missing`
└ @ CSV C:\Users\Lyy\.julia\packages\CSV\XLcqT\src\file.jl:592
┌ Warning: thread = 1 warning: only found 6 / 7 columns around data row: 6. Filling remaining columns with `missing`
└ @ CSV C:\Users\Lyy\.julia\packages\CSV\XLcqT\src\file.jl:592


OPTIMAL

Expected cost = 17720.0

--- Period 1 ---
Biomass         : 0.0
Hydroelectric   : 0.0
Geothermal      : 0.0
NG_CCGT         : 220.0
NG_CT           : 100.0
Wind            : 0.0
Solar           : 195.0
Solar1 = 450.0  (max =450.0)
Wind1  = 135.0  (max =135.0)

--- Scenario s1  prob=0.525 ---
Solar2 = 475.0  (max=475.0)
Wind2  = 120.0  (max=120.0)
Biomass         : 0.0
Hydroelectric   : 0.0
Geothermal      : 0.0
NG_CCGT         : 220.0
NG_CT           : 100.0
Wind            : 0.0
Solar           : 285.0

--- Scenario s2  prob=0.225 ---
Solar2 = 375.0  (max=375.0)
Wind2  = 150.0  (max=150.0)
Biomass         : 0.0
Hydroelectric   : 0.0
Geothermal      : 0.0
NG_CCGT         : 220.0
NG_CT           : 100.0
Wind            : 0.0
Solar           : 355.0

--- Scenario s3  prob=0.175 ---
Solar2 = 475.0  (max=475.0)
Wind2  = 120.0  (max=120.0)
Biomass         : 0.0
Hydroelectric   : 0.0
Geothermal      : 0.0
NG_CCGT         : 220.0
NG_CT           : 100.0
Wind            : 85.0
Solar  

## References
Google.